# FastAPI Error Handling and Testing

## Objectives
1. Test API connection.
2. Test invalid prediction requests.
3. Test missing input fields.
4. Test invalid data types.
5. Test RAG endpoint failures.
6. Test HTTP status codes.
7. Verify API error messages.

In [1]:
import requests
import json
import time

BASE_URL = "http://127.0.0.1:8000"

print("Libraries imported successfully!")
print("API URL:", BASE_URL)

Libraries imported successfully!
API URL: http://127.0.0.1:8000


In [2]:
try:
    response = requests.get(BASE_URL, timeout=10)

    print("API Status Code:", response.status_code)
    print("API Response:", response.text)

except requests.exceptions.ConnectionError:
    print("ERROR: FastAPI server is not running.")

except requests.exceptions.Timeout:
    print("ERROR: API request timed out.")

except requests.exceptions.RequestException as error:
    print("API Error:", error)

API Status Code: 200
API Response: {"message":"Welcome to Sales & Operations Intelligence Copilot","status":"Backend is running"}


In [3]:
try:
    response = requests.post(
        f"{BASE_URL}/predict",
        json={},
        timeout=30
    )

    print("Status Code:", response.status_code)
    print("Response:", response.text)

except requests.exceptions.RequestException as error:
    print("Request Error:", error)

Status Code: 422
Response: {"detail":[{"type":"missing","loc":["body","lead_source"],"msg":"Field required","input":{}},{"type":"missing","loc":["body","industry"],"msg":"Field required","input":{}},{"type":"missing","loc":["body","company_size"],"msg":"Field required","input":{}},{"type":"missing","loc":["body","budget"],"msg":"Field required","input":{}},{"type":"missing","loc":["body","website_visits"],"msg":"Field required","input":{}},{"type":"missing","loc":["body","emails_opened"],"msg":"Field required","input":{}},{"type":"missing","loc":["body","calls_made"],"msg":"Field required","input":{}},{"type":"missing","loc":["body","previous_interactions"],"msg":"Field required","input":{}},{"type":"missing","loc":["body","response_time_hours"],"msg":"Field required","input":{}}]}


In [4]:
try:
    response = requests.post(
        f"{BASE_URL}/ask",
        json={},
        timeout=60
    )

    print("Status Code:", response.status_code)
    print("Response:", response.text)

except requests.exceptions.RequestException as error:
    print("Request Error:", error)

Status Code: 422
Response: {"detail":[{"type":"missing","loc":["body","question"],"msg":"Field required","input":{}}]}


In [5]:
try:
    response = requests.post(
        f"{BASE_URL}/predict",
        json="invalid data",
        timeout=30
    )

    print("Status Code:", response.status_code)
    print("Response:", response.text)

except requests.exceptions.RequestException as error:
    print("Request Error:", error)

Status Code: 422
Response: {"detail":[{"type":"model_attributes_type","loc":["body"],"msg":"Input should be a valid dictionary or object to extract fields from","input":"invalid data"}]}


In [6]:
try:
    response = requests.post(
        f"{BASE_URL}/predict",
        json={
            "budget": 50000
        },
        timeout=30
    )

    print("Status Code:", response.status_code)
    print("Response:", response.text)

except requests.exceptions.RequestException as error:
    print("Request Error:", error)

Status Code: 422
Response: {"detail":[{"type":"missing","loc":["body","lead_source"],"msg":"Field required","input":{"budget":50000}},{"type":"missing","loc":["body","industry"],"msg":"Field required","input":{"budget":50000}},{"type":"missing","loc":["body","company_size"],"msg":"Field required","input":{"budget":50000}},{"type":"missing","loc":["body","website_visits"],"msg":"Field required","input":{"budget":50000}},{"type":"missing","loc":["body","emails_opened"],"msg":"Field required","input":{"budget":50000}},{"type":"missing","loc":["body","calls_made"],"msg":"Field required","input":{"budget":50000}},{"type":"missing","loc":["body","previous_interactions"],"msg":"Field required","input":{"budget":50000}},{"type":"missing","loc":["body","response_time_hours"],"msg":"Field required","input":{"budget":50000}}]}


In [7]:
invalid_payload = {
    "lead_source": "Website",
    "industry": "IT",
    "company_size": "Medium",
    "budget": -50000,
    "website_visits": 20,
    "emails_opened": 8,
    "calls_made": 5,
    "previous_interactions": 3,
    "response_time_hours": 12
}

try:
    response = requests.post(
        f"{BASE_URL}/predict",
        json=invalid_payload,
        timeout=30
    )

    print("Status Code:", response.status_code)
    print("Response:", response.text)

except requests.exceptions.RequestException as error:
    print("Request Error:", error)

Status Code: 422
Response: {"detail":[{"type":"greater_than_equal","loc":["body","budget"],"msg":"Input should be greater than or equal to 0","input":-50000,"ctx":{"ge":0.0}}]}


In [8]:
invalid_payload = {
    "lead_source": "Website",
    "industry": "IT",
    "company_size": "Medium",
    "budget": "fifty thousand",
    "website_visits": 20,
    "emails_opened": 8,
    "calls_made": 5,
    "previous_interactions": 3,
    "response_time_hours": 12
}

try:
    response = requests.post(
        f"{BASE_URL}/predict",
        json=invalid_payload,
        timeout=30
    )

    print("Status Code:", response.status_code)
    print("Response:", response.text)

except requests.exceptions.RequestException as error:
    print("Request Error:", error)

Status Code: 422
Response: {"detail":[{"type":"float_parsing","loc":["body","budget"],"msg":"Input should be a valid number, unable to parse string as a number","input":"fifty thousand"}]}


In [9]:
invalid_payload = {
    "lead_source": "Alien Portal",
    "industry": "IT",
    "company_size": "Medium",
    "budget": 50000,
    "website_visits": 20,
    "emails_opened": 8,
    "calls_made": 5,
    "previous_interactions": 3,
    "response_time_hours": 12
}

try:
    response = requests.post(
        f"{BASE_URL}/predict",
        json=invalid_payload,
        timeout=30
    )

    print("Status Code:", response.status_code)
    print("Response:", response.text)

except requests.exceptions.RequestException as error:
    print("Request Error:", error)

Status Code: 200
Response: {"conversion_probability":0.7677,"prediction":"Likely to Convert","threshold":0.75}


In [10]:
try:
    response = requests.post(
        f"{BASE_URL}/ask",
        json={"question": ""},
        timeout=60
    )

    print("Status Code:", response.status_code)
    print("Response:", response.text)

except requests.exceptions.RequestException as error:
    print("Request Error:", error)

Status Code: 422
Response: {"detail":[{"type":"string_too_short","loc":["body","question"],"msg":"String should have at least 3 characters","input":"","ctx":{"min_length":3}}]}


In [11]:
try:
    response = requests.post(
        f"{BASE_URL}/ask",
        json={
            "question": "What services does the company offer?"
        },
        timeout=120
    )

    print("Status Code:", response.status_code)
    print("Response:")

    try:
        print(json.dumps(response.json(), indent=4))
    except ValueError:
        print(response.text)

except requests.exceptions.RequestException as error:
    print("Request Error:", error)

Status Code: 200
Response:
{
    "question": "What services does the company offer?",
    "answer": "The company offers the following services:\n\n1. Digital Marketing\n2. Social Media Marketing\n3. Lead Generation\n4. Business Development\n5. Content Marketing\n6. Branding and Creative Design.\n\nThese services are mentioned in the company's [Source 1] and [Source 2] documents.",
    "citations": [
        {
            "citation": "Source 1",
            "filename": "company_profile.txt",
            "chunk_id": 1,
            "distance": 0.5714,
            "supporting_text": "Our Services:\n- Digital Marketing\n- Social Media Marketing\n- Lead Generation\n- Business Development\n- Content Marketing\n- Branding and Creative Design\n\nTarget Customers:\nSmall businesses, startups, interior companies, manufacturing businesses, and service providers.\n\nCustomer Communication:\nCustomers can contact the company through phone, email, or WhatsApp."
        },
        {
            "citat

In [12]:
try:
    response = requests.post(
        f"{BASE_URL}/ask",
        json={
            "question": "What is the company's annual revenue and net profit?"
        },
        timeout=120
    )

    print("Status Code:", response.status_code)
    print("Response:")

    print(json.dumps(response.json(), indent=4))

except requests.exceptions.RequestException as error:
    print("Request Error:", error)

Status Code: 200
Response:
{
    "question": "What is the company's annual revenue and net profit?",
    "answer": "I'm unable to provide the company's annual revenue and net profit as this information is not available in the provided context.",
    "citations": [
        {
            "citation": "Source 1",
            "filename": "products_and_services.txt",
            "chunk_id": 2,
            "distance": 0.6166,
            "supporting_text": "Pricing:\nPricing depends on project requirements, scope, and duration. Customers should contact the company for a quotation."
        },
        {
            "citation": "Source 2",
            "filename": "company_profile.txt",
            "chunk_id": 0,
            "distance": 0.792,
            "supporting_text": "COMPANY PROFILE\n\nCompany Name: Triads Future\nBusiness Type: Digital Marketing and Business Development Agency\nLocation: Thiruvananthapuram, Kerala, India\n\nAbout Us:\nTriads Future helps businesses grow through digital 

In [13]:
response = requests.get(
    f"{BASE_URL}/health",
    timeout=10
)

print("Status Code:", response.status_code)
print("Response:", response.json())

Status Code: 200
Response: {'status': 'healthy', 'application': 'Sales & Operations Intelligence Copilot', 'version': '1.0.0', 'ml_model_loaded': True}


In [16]:
valid_payload = {
    "lead_source": "Website",
    "industry": "IT",
    "company_size": "Medium",
    "budget": 50000,
    "website_visits": 20,
    "emails_opened": 8,
    "calls_made": 5,
    "previous_interactions": 3,
    "response_time_hours": 12
}

response = requests.post(
    f"{BASE_URL}/predict",
    json=valid_payload,
    timeout=30
)

print("Status Code:", response.status_code)
print("Response:", response.json())

Status Code: 200
Response: {'conversion_probability': 0.7898, 'prediction': 'Likely to Convert', 'threshold': 0.75}


In [17]:
response = requests.get(
    f"{BASE_URL}/health",
    timeout=10
)

print("Status Code:", response.status_code)
print("Response:", response.json())

Status Code: 200
Response: {'status': 'healthy', 'application': 'Sales & Operations Intelligence Copilot', 'version': '1.0.0', 'ml_model_loaded': True}


In [18]:
response = requests.post(
    f"{BASE_URL}/test-rag-error",
    timeout=10
)

print("Status Code:", response.status_code)
print("Response:", response.json())

Status Code: 503
Response: {'detail': 'AI assistant is temporarily unavailable. Please try again.'}


In [19]:
response = requests.get(f"{BASE_URL}/health")
print(response.status_code)
print(response.json())

200
{'status': 'healthy', 'application': 'Sales & Operations Intelligence Copilot', 'version': '1.0.0', 'ml_model_loaded': True}


In [20]:
response = requests.post(
    f"{BASE_URL}/predict",
    json=valid_payload
)

print(response.status_code)
print(response.json())

200
{'conversion_probability': 0.7898, 'prediction': 'Likely to Convert', 'threshold': 0.75}


In [21]:
response = requests.post(
    f"{BASE_URL}/ask",
    json={"question": "What services does the company offer?"},
    timeout=120
)

print(response.status_code)
print(response.json())

200
{'question': 'What services does the company offer?', 'answer': 'According to the company context, Triads Future offers the following services:\n\n1. Digital Marketing\n2. Social Media Marketing\n3. Lead Generation\n4. Business Development\n5. Content Marketing\n6. Branding and Creative Design\n\nThese services are mentioned in [Source 1] and [Source 2].', 'citations': [{'citation': 'Source 1', 'filename': 'company_profile.txt', 'chunk_id': 1, 'distance': 0.5714, 'supporting_text': 'Our Services:\n- Digital Marketing\n- Social Media Marketing\n- Lead Generation\n- Business Development\n- Content Marketing\n- Branding and Creative Design\n\nTarget Customers:\nSmall businesses, startups, interior companies, manufacturing businesses, and service providers.\n\nCustomer Communication:\nCustomers can contact the company through phone, email, or WhatsApp.'}, {'citation': 'Source 2', 'filename': 'products_and_services.txt', 'chunk_id': 0, 'distance': 0.6317, 'supporting_text': 'PRODUCTS AN